# Evolution du TLS

Parametres dans les conventions D, sigma_k et x_tls. Le couplage est `sqrt`.
Les fonctions de calcul vivent dans `experiment/`; ce notebook sert au pilotage.
Voir [la documentation](../documentation/main.pdf).


In [ ]:
from pathlib import Path
import sys

# Works whether the kernel starts at the repository root or inside notebooks/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src' / 'xp_config.py').is_file() and (p / 'experiment').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
from src.xp_config import ExperimentConfig
from src.experiment import estimate_config
from src.fidelities import fidelities_over_time


## Parametres


In [ ]:
CTRL_M_EXPLICIT = True
M = 5                         # Must be odd; k=0 is included.
cutoffs = {'ir_cutoff': 0.0, 'uv_cutoff': 2.0}  # Used only when control is False.

param_atom = {'omega_0': 1.0, 'D': 0.2, 'L': 2*np.pi,
              'x_tls': 0.0, 'coupling': 'sqrt', 'initial_state': 'g'}
param_photon = {'k_0': 1.0, 'sigma_k': 0.4, 'x_0': -2.0,
                'state': 'number', 'n': 1, 'alpha': 1.0}
param_time_evol = {'T': 4.0, 'dt': 0.05, 'rtol': 1e-9, 'atol': 1e-11}
n_max = 2
truncation = 'full+totalcap'
RWA = False

config = ExperimentConfig(param_photon, param_atom, param_time_evol, cutoffs,
                          n_max=n_max, truncation=truncation, RWA=RWA,
                          CTRL_M_EXPLICIT=CTRL_M_EXPLICIT, M=M)


## Estimation et calcul


In [ ]:
from experiment.atom_evolution import run_atom_evolution, plot_atom_evolution
schemes = ('full+totalcap', 'truncated')
config = replace(config, param_atom={**param_atom, 'initial_state': 'e'},
                 param_photon={**param_photon, 'n': 0})
checks = [estimate_config(replace(config, truncation=s)) for s in schemes]
if not all(check['feasible'] for check in checks):
    raise MemoryError('Adjust the parameters after reviewing the resource estimates.')
runs = run_atom_evolution(config, schemes)
fig = plot_atom_evolution(runs)
plt.show()


## Diagnostics complementaires


In [ ]:
F = fidelities_over_time(runs['truncated'], runs['full+totalcap'])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, ('F_state', 'F_atom')):
    ax.plot(runs['truncated'].times, F[name])
    ax.set(xlabel='t', ylabel=name)
    print(name, 'initial:', F[name][0], 'time mean:', np.mean(F[name]))
plt.show()
